## 0. Load packages and CSVs

In [2]:
import pandas as pd
import numpy as np
import re

# Load the CSVs
agendapunt = pd.read_csv("Agendapunt.csv")
besluit = pd.read_csv("Besluit.csv")
activiteit = pd.read_csv("Activiteit.csv")

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## 1. Column: `Id`
This is `Id` from `Agendapunt.csv`.

In [3]:
col_id = agendapunt["Id"].astype(str)

## 2. Column: `Topic`
This is `Onderwerp` from `Agendapunt.csv`.

In [4]:
col_topic = agendapunt["Onderwerp"].astype(str)

## 3. Column: `Decision`

- `Besluit.csv` contains `Agendapunt_Id`
- match that to `Agendapunt.csv["Id]`
- use `BesluitSoort` from `Besluit.csv`
- map:
    - `"Stemmen - aangenomen"` → `"Accepted"`
    - `"Stemmen - ingetrokken"` or `"Stemmen - verworpen"` → `"Rejected"`


In [5]:
# Keep only the columns we need
besluit_subset = besluit[["Agendapunt_Id", "BesluitSoort"]].copy()

# If there are multiple rows per Agendapunt_Id, keep the first one
besluit_subset = besluit_subset.drop_duplicates(subset="Agendapunt_Id", keep="first")

# Merge decision info onto agendapunt
decision_merge = agendapunt[["Id"]].merge(
    besluit_subset,
    how="left",
    left_on="Id",
    right_on="Agendapunt_Id"
)

# Map decision labels
decision_map = {
    "Stemmen - aangenomen": "Accepted",
    "Stemmen - ingetrokken": "Rejected",
    "Stemmen - verworpen": "Rejected"
}

col_decision = decision_merge["BesluitSoort"].map(decision_map)


## 4. Column: `Activiteit_Id`


In [6]:
col_activiteit_id = agendapunt["Activiteit_Id"]

## 5. Prepare `Activiteit.csv` helper columns
Before engineering the remaining features it helps to preprocess the datetime fields.

In [7]:
activiteit_work = activiteit.copy()

# Parse datetime columns
activiteit_work["Aanvangstijd_dt"] = pd.to_datetime(
    activiteit_work["Aanvangstijd"],
    errors="coerce",
    utc=True
)

activiteit_work["Eindtijd_dt"] = pd.to_datetime(
    activiteit_work["Eindtijd"],
    errors="coerce",
    utc=True
)

activiteit_work["Datum_dt"] = pd.to_datetime(
    activiteit_work["Datum"],
    errors="coerce",
    utc=True
)

In [8]:
# Extract date as YYYY-MM-DD string
activiteit_work["Datum_date"] = activiteit_work["Datum"].astype(str).str[:10]

# Extract raw time from Aanvangstijd
activiteit_work["Aanvangstijd_time"] = activiteit_work["Aanvangstijd_dt"].dt.time

In [9]:
# Lookup table
activiteit_lookup = activiteit_work.set_index("Id")

## 6. Column: `Topic_duration`
We want:
- `Eindtijd - Aanvangstijd`
- for the corresponding `Activiteit_Id`

In [10]:
# Map the activity start and end times onto the topic rows
topic_start = col_activiteit_id.map(activiteit_lookup["Aanvangstijd_dt"])
topic_end = col_activiteit_id.map(activiteit_lookup["Eindtijd_dt"])

# Duration as timedelta
col_topic_duration = topic_end - topic_start

In [11]:
# In minutes
col_topic_duration_minutes = (topic_end - topic_start).dt.total_seconds() / 60

## 7. Column: `Topic_date`
We want the first 10 characters of `Datum`, so `YYYY-MM-DD`.

In [12]:
col_topic_date = col_activiteit_id.map(activiteit_lookup["Datum_date"])

## 8. Column: `Topic_density_per_day`
We want the count of how many rows in `Activiteit.csv` have the same date.

So first count activities per date, then map that count to each topic row using `Topic_date`.

In [13]:
density_per_day = activiteit_work["Datum_date"].value_counts()

col_topic_density_per_day = col_topic_date.map(density_per_day)

## 9. Column: `Time_of_day_time`
This is the raw time from `Aanvangstijd`.

In [14]:
col_time_of_day_time = col_activiteit_id.map(activiteit_lookup["Aanvangstijd_time"])

## 10. Column: `Time_of_day_category`
Classify the raw times into buckets:
- 05:00 - 08:00: `"Early morning"`
- 08:00 - 12:00: `"Late morning"`
- 12:00 - 15:00: `"Early afternoon"`
- 15:00 - 18:00: `"Late afternoon"`
- 18:00 - 22:00: `"Evening"`
- 22:00 - 05:00: `"Night"`

In [15]:
def categorize_time_of_day(t):
    if pd.isna(t):
        return np.nan
    
    hour = t.hour
    
    if 5 <= hour < 8:
        return "Early morning"
    elif 8 <= hour < 12:
        return "Late morning"
    elif 12 <= hour < 15:
        return "Early afternoon"
    elif 15 <= hour < 18:
        return "Late afternoon"
    elif 18 <= hour < 22:
        return "Evening"
    else:
        return "Night"
    
col_time_of_day_category = col_time_of_day_time.apply(categorize_time_of_day)

## 11. Column: `Topic_category`

In [16]:
import pandas as pd
import re

category_keywords = {
    "Healthcare": [
        "zorg", "gezondheid", "gezond", "ggd", "jeugdzorg", "ouderenzorg",
        "thuiszorg", "verpleging", "ziekenhuis", "medisch", "huisarts",
        "ggz", "welzijnszorg", "publieke gezondheid", "volksgezondheid",
        "zorginstelling", "zorgkosten", "preventie", "gezondheidszorg"
    ],
    
    "Education": [
        "onderwijs", "school", "scholen", "basisschool", "middelbare school",
        "leerling", "leerlingen", "student", "studenten", "docent", "leraar",
        "jeugd", "jongeren", "kinderopvang", "opvang", "onderwijshuisvesting",
        "voorschool", "jeugdhulp", "jeugdbeleid", "studie"
    ],
    
    "Housing": [
        "wonen", "woning", "woningen", "huisvesting", "huur", "huurwoning",
        "koopwoning", "woningbouw", "woonbeleid", "woonruimte", "dakloos",
        "dakloosheid", "buurt", "wijk", "gebiedsontwikkeling", "bestemmingsplan",
        "ruimtelijke ordening", "omgevingsplan", "omgevingswet", "bouwproject",
        "nieuwbouw", "vastgoed", "pand", "panden", "leefomgeving"
    ],
    
    "Infrastructure & Transport": [
        "verkeer", "vervoer", "mobiliteit", "fiets", "fietsen", "fietspad",
        "auto", "autos", "parkeren", "parkeer", "ov", "openbaar vervoer",
        "tram", "bus", "metro", "trein", "station", "weg", "wegen",
        "straat", "brug", "verkeersveiligheid", "doorstroming", "bereikbaarheid"
    ],
    
    "Environment & Climate": [
        "klimaat", "milieu", "duurzaam", "duurzaamheid", "co2", "stikstof",
        "luchtkwaliteit", "waterkwaliteit", "afval", "recycling", "circulair",
        "circulaire economie", "groen", "natuur", "biodiversiteit",
        "energietransitie", "klimaatadaptatie", "verduurzaming", "milieubeleid",
        "vervuiling", "uitstoot"
    ],
    
    "Economy & Finance": [
        "economie", "economisch", "financien", "financiën", "begroting",
        "budget", "subsidie", "subsidies", "belasting", "belastingen",
        "heffing", "heffingen", "kosten", "baten", "uitgaven", "inkomsten",
        "gemeentefonds", "investering", "investeringen", "onderneming",
        "ondernemers", "mkb", "markt", "aanbesteding"
    ],
    
    "Employment & Labour": [
        "werk", "arbeid", "baan", "banen", "werkgelegenheid", "werkloosheid",
        "inkomen", "uitkering", "bijstand", "minimumloon", "arbeidsmarkt",
        "re-integratie", "participatiewet", "werkzoekenden", "werkenden"
    ],
    
    "Social Welfare": [
        "sociaal", "sociale zaken", "participatie", "inclusie", "armoede",
        "schuld", "schulden", "schuldhulp", "gelijke kansen", "ongelijkheid",
        "voorziening", "maatschappelijke ondersteuning", "wmo", "kwetsbare groepen",
        "toegankelijkheid", "participeren", "burgerparticipatie", "welzijn"
    ],
    
    "Public Safety & Policing": [
        "veiligheid", "handhaving", "politie", "brandweer", "criminaliteit",
        "overlast", "openbare orde", "toezicht", "boa", "noodhulp",
        "incident", "misdaad", "geweld", "ondermijning", "drugsoverlast",
        "brandveiligheid", "rampenbestrijding", "crisisbeheersing"
    ],
    
    "Governance & Administration": [
        "bestuur", "college", "raad", "raadsvergadering", "gemeenteraad",
        "burgemeester", "wethouder", "motie", "amendement", "verordening",
        "beleid", "beleidsplan", "uitvoering", "overheid", "gemeente",
        "provincie", "minister", "ministerie", "regeling", "wet", "wetgeving",
        "toezegging", "brief", "besluit", "agenda", "administratie"
    ],
    
    "Immigration": [
        "migratie", "migrant", "migranten", "vluchteling", "vluchtelingen",
        "asiel", "statushouder", "inburgering", "integratie", "nieuwkomer",
        "nieuwkomers", "opvanglocatie", "azc", "arbeidsmigranten"
    ],
    
    "Culture & Recreation": [
        "cultuur", "cultureel", "kunst", "museum", "theater", "bibliotheek",
        "erfgoed", "monument", "festival", "evenement", "sport", "sporten",
        "sportvereniging", "recreatie", "vrije tijd", "park", "speeltuin"
    ],
    
    "Technology & Digitalisation": [
        "digitalisering", "digitaal", "ict", "data", "algoritme", "ai",
        "kunstmatige intelligentie", "cyber", "cybersecurity", "privacy",
        "informatieveiligheid", "technologie", "automatisering", "software",
        "systeem", "online", "internet", "platform"
    ],
    
    "Energy": [
        "energie", "netcongestie", "elektriciteit", "warmte", "gas",
        "laadinfrastructuur", "stroom", "zonnepanelen", "windenergie",
        "warmtenet", "energienet", "infrastructuur", "riolering", "waterleiding",
        "nutsvoorziening", "capaciteit", "netbeheer"
    ]
}

In [17]:
def categorize_topic(topic):
    if pd.isna(topic):
        return "Overig"
    
    topic_clean = str(topic).lower()
    topic_clean = re.sub(r"[^\w\s-]", " ", topic_clean)
    topic_clean = re.sub(r"\s+", " ", topic_clean).strip()
    
    scores = {}
    
    for category, keywords in category_keywords.items():
        score = 0
        for kw in keywords:
            if kw in topic_clean:
                score += len(kw.split()) + 1
        if score > 0:
            scores[category] = score
    
    if scores:
        return max(scores, key=scores.get)
    return "Other"

## 12. Combine all columns into the final dataframe

In [22]:
final_df = pd.DataFrame({
    "Id": col_id,
    "Topic": col_topic,
    "Decision": col_decision,
    "Activiteit_Id": col_activiteit_id,
    "Topic_duration": col_topic_duration,
    "Topic_duration_minutes": col_topic_duration_minutes,
    "Topic_date": col_topic_date,
    "Topic_density_per_day": col_topic_density_per_day,
    "Time_of_day_time": col_time_of_day_time.astype(str),
    "Time_of_day_category": col_time_of_day_category
})

final_df = final_df.dropna(subset=["Decision"])
final_df = final_df[final_df["Topic_duration_minutes"].notna() & (final_df["Topic_duration_minutes"] >= 0)]

In [37]:
# Topic category column
final_df["Topic_category"] = final_df["Topic"].apply(categorize_topic)

# Meeting type column
final_df["Meeting_type"] = np.where(
    final_df["Topic_duration_minutes"] <= 100,
    "Plenair",
    "Commissie"
)

# Season column
final_df["Season"] = np.select(
    [
        final_df["Topic_date"].str[5:7].isin(["12", "01", "02"]),
        final_df["Topic_date"].str[5:7].isin(["03", "04", "05"]),
        final_df["Topic_date"].str[5:7].isin(["06", "07", "08"]),
        final_df["Topic_date"].str[5:7].isin(["09", "10", "11"]),
    ],
    [
        "Winter",
        "Spring",
        "Summer",
        "Autumn"
    ],
    default=np.nan
)



In [38]:
final_df = pd.DataFrame({
    "Id": col_id,
    "Topic": col_topic,
    "Topic_category": final_df["Topic_category"],
    "Decision": col_decision,
    "Activiteit_Id": col_activiteit_id,
    "Meeting_type": final_df["Meeting_type"],
    "Topic_duration": col_topic_duration,
    "Topic_duration_minutes": col_topic_duration_minutes,
    "Topic_date": col_topic_date,
    "Season": final_df["Season"],
    "Topic_density_per_day": col_topic_density_per_day,
    "Time_of_day_time": col_time_of_day_time.astype(str),
    "Time_of_day_category": col_time_of_day_category
})

final_df = final_df.dropna(subset=["Decision"])
final_df = final_df[final_df["Topic_duration_minutes"].notna() & (final_df["Topic_duration_minutes"] >= 0)]

## 12. Inspect results

In [39]:
final_df.head()

,Id,Topic,Topic_category,Decision,Activiteit_Id,Meeting_type,Topic_duration,Topic_duration_minutes,Topic_date,Season,Topic_density_per_day,Time_of_day_time,Time_of_day_category
23,bef9fca9-e2c7-4a1a-badc-0005b3960051,Ingetrokken amendement,Governance & Administration,Rejected,19e5aad0-2e78-4645-a3ab-91087d3b2d10,Commissie,0 days 10:20:00,620.0,2018-11-22,Autumn,43.0,10:45:00,Late morning
140,c295ac85-a764-453e-a3db-002151e8f946,Moties ingediend bij Vaststelling van de begro...,Governance & Administration,Rejected,978d15c6-81d4-4d85-b544-a0043c1fbbfc,Commissie,0 days 08:44:00,524.0,2014-01-28,Winter,19.0,14:15:00,Early afternoon
234,1ed48294-fd14-4f85-bca7-0030a3653139,Moties ingediend bij het VSO Ontwerpbesluit gr...,Governance & Administration,Accepted,61de1e1c-6136-4f7a-a4f6-03e91979dc86,Commissie,0 days 08:44:00,524.0,2014-06-03,Summer,16.0,13:15:00,Early afternoon
268,de378c8f-2513-4a96-844f-00372d1cc718,moties ingediend bij het tweeminutendebat Zoön...,Governance & Administration,Accepted,78708110-1ddd-48dd-a708-d60537f376f2,Plenair,0 days 00:50:00,50.0,2022-03-22,Spring,27.0,14:10:00,Early afternoon
307,cc7649b8-9643-4f4f-92e4-003e674d00b0,Moties ingediend bij Wijziging van de Wet educ...,Governance & Administration,Rejected,c153109b-69fa-42f1-be25-e15b49a52331,Plenair,0 days 00:30:00,30.0,2022-05-31,Spring,24.0,13:00:00,Early afternoon


In [40]:
final_df.info()

<class 'pandas.DataFrame'>
Index: 9055 entries, 23 to 333397
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype          
---  ------                  --------------  -----          
 0   Id                      9055 non-null   str            
 1   Topic                   9055 non-null   str            
 2   Topic_category          9055 non-null   str            
 3   Decision                9055 non-null   str            
 4   Activiteit_Id           9055 non-null   str            
 5   Meeting_type            9055 non-null   str            
 6   Topic_duration          9055 non-null   timedelta64[us]
 7   Topic_duration_minutes  9055 non-null   float64        
 8   Topic_date              9055 non-null   str            
 9   Season                  9055 non-null   str            
 10  Topic_density_per_day   9055 non-null   float64        
 11  Time_of_day_time        9055 non-null   str            
 12  Time_of_day_category    9055 non-null   str    

## 13. Save to CSV


In [41]:
final_df.to_csv("engineered_topics2.csv", index=False)